# 4단계: 우선순위 도출

**목적:** 1~3단계 산출 지수를 가중합산하여 자율주행 택시 확대 우선순위를 도출하고,
결과 검증(잠재 수요 역산)으로 타당성 제시

**복합지수 공식:**
```
우선순위 지수 = 오피스밀집도(35%) + 대중교통공백(35%) + 야간택시수요(30%)
```

**산출물:** 행정동별 우선순위 지수 랭킹 + Choropleth 지도 + 결과 검증 표

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = 'output/'
DATA_DIR   = 'data/'
print('라이브러리 로드 완료')


라이브러리 로드 완료


## 1. 이전 단계 결과 로드

In [2]:
# ── 1~3단계 산출물 로드
df_stage1 = pd.read_csv(OUTPUT_DIR + 'stage01_office_index.csv',   dtype={'행정동코드': str})
df_stage2 = pd.read_csv(OUTPUT_DIR + 'stage02_transport_gap.csv',  dtype={'행정동코드': str})
df_stage3 = pd.read_csv(OUTPUT_DIR + 'stage03_taxi_demand.csv',    dtype={'행정동코드': str})

# 행정동코드 8자리 통일
for df in [df_stage1, df_stage2, df_stage3]:
    df['행정동코드'] = df['행정동코드'].str.zfill(8)

print('1단계:', df_stage1.shape, '/ 컬럼:', list(df_stage1.columns))
print('2단계:', df_stage2.shape, '/ 컬럼:', list(df_stage2.columns))
print('3단계:', df_stage3.shape, '/ 컬럼:', list(df_stage3.columns))


1단계: (426, 6) / 컬럼: ['행정동코드', '행정동명', '오피스_종사자수', '전체_종사자수', '오피스_비율', '오피스_밀집도_지수']
2단계: (426, 5) / 컬럼: ['행정동코드', '행정동명', 'owl_covered', 'owl_coverage_ratio', '대중교통공백_지수']
3단계: (424, 4) / 컬럼: ['행정동코드', '행정동명', '심야유동인구', '야간수요_지수']


In [3]:
# ── 행정동 경계 로드
gdf_dong = gpd.read_file(DATA_DIR + 'bnd_dong_11_2025_2Q.shp')
if gdf_dong.crs is None or gdf_dong.crs.to_epsg() != 4326:
    gdf_dong = gdf_dong.to_crs('EPSG:4326')
gdf_dong = gdf_dong.rename(columns={'ADM_CD': '행정동코드', 'ADM_NM': '행정동명'})
gdf_dong['행정동코드'] = gdf_dong['행정동코드'].astype(str).str.zfill(8)

# ── 3개 지수 병합 (컬럼명: 대중교통공백_지수 / 야간수요_지수)
df = gdf_dong[['행정동코드', '행정동명', 'geometry']].copy()
df = df.merge(df_stage1[['행정동코드', '오피스_종사자수', '오피스_밀집도_지수']], on='행정동코드', how='left')
df = df.merge(df_stage2[['행정동코드', '대중교통공백_지수']],   on='행정동코드', how='left')
df = df.merge(df_stage3[['행정동코드', '심야유동인구', '야간수요_지수']], on='행정동코드', how='left')
df = df.fillna(0)

print(f'병합 완료: {len(df)}개 행정동')
print('결측치 확인:')
print(df[['오피스_밀집도_지수','대중교통공백_지수','야간수요_지수']].isnull().sum())


병합 완료: 426개 행정동
결측치 확인:
오피스_밀집도_지수    0
대중교통공백_지수     0
야간수요_지수       0
dtype: int64


## 2. 최종 Min-Max 정규화 및 가중합산

In [4]:
# ── 최종 정규화 (각 지수가 이미 0~1이지만 재정규화로 스케일 통일)
scaler = MinMaxScaler()
features = ['오피스_밀집도_지수', '대중교통공백_지수', '야간수요_지수']
df[features] = scaler.fit_transform(df[features])

# ── 가중합산 복합지수
W_OFFICE    = 0.35   # 오피스 밀집도 35%
W_TRANSPORT = 0.35   # 대중교통 공백 35%
W_DEMAND    = 0.30   # 야간 택시 수요 30%

df['우선순위_지수'] = (
    df['오피스_밀집도_지수'] * W_OFFICE +
    df['대중교통공백_지수']  * W_TRANSPORT +
    df['야간수요_지수']      * W_DEMAND
)

# 순위 부여
df['순위'] = df['우선순위_지수'].rank(ascending=False, method='min').astype(int)

# 상위 20개 출력
top20 = df.sort_values('우선순위_지수', ascending=False).head(20)
print('=== 자율주행 택시 확대 우선순위 TOP 20 ===')
print(top20[['순위','행정동명','오피스_밀집도_지수','대중교통공백_지수','야간수요_지수','우선순위_지수']].to_string(index=False))


=== 자율주행 택시 확대 우선순위 TOP 20 ===
 순위    행정동명  오피스_밀집도_지수  대중교통공백_지수  야간수요_지수  우선순위_지수
  1     여의동    1.000000   0.743700 0.149755 0.655221
  2     가산동    0.781204   0.690931 0.179907 0.569220
  3    역삼1동    0.773284   0.231402 0.712228 0.565308
  4    논현2동    0.265958   1.000000 0.391119 0.560421
  5    수유2동    0.003212   1.000000 0.446271 0.485006
  6     성현동    0.003175   0.988677 0.455016 0.483653
  7    삼각산동    0.002171   1.000000 0.430229 0.479829
  8     삼전동    0.018561   0.992517 0.402672 0.474679
  9    개봉3동    0.040078   0.935690 0.415043 0.466032
 10 금호2·3가동    0.001980   0.989320 0.393645 0.465049
 11     신사동    0.060978   0.968972 0.344431 0.463812
 12    신길3동    0.003175   1.000000 0.374827 0.463559
 13    중계1동    0.002625   1.000000 0.369738 0.461840
 14    삼성2동    0.235846   0.691353 0.448957 0.459207
 15    압구정동    0.063530   1.000000 0.289620 0.459122
 16    방배본동    0.023071   0.969495 0.370311 0.458492
 17    신길6동    0.004136   1.000000 0.342332 0.454147
 18   금호1가동    

## 3. 우선순위 지도 시각화 (Choropleth)

In [5]:
# ── Choropleth 지도 (geopandas PNG — 반출 가능)
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# 왼쪽: 복합 우선순위 지수
df.plot(
    column='우선순위_지수',
    ax=axes[0],
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Priority Index (0~1)', 'shrink': 0.7},
    missing_kwds={'color': 'lightgrey'}
)

# TOP 5 라벨 표시
top5 = df.sort_values('우선순위_지수', ascending=False).head(5)
for _, row in top5.iterrows():
    cx = row['geometry'].centroid.x
    cy = row['geometry'].centroid.y
    axes[0].annotate(f"{row['순위']}.{row['행정동명']}", xy=(cx, cy),
                     fontsize=6, ha='center', color='black',
                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

axes[0].set_title('Autonomous Taxi Priority Index by Dong')
axes[0].set_axis_off()

# 오른쪽: 3개 지수 산점도 (오피스 vs 야간수요, 색=교통공백)
scatter = axes[1].scatter(
    df['오피스_밀집도_지수'],
    df['야간수요_지수'],
    c=df['대중교통공백_지수'],
    s=df['우선순위_지수'] * 100 + 10,
    cmap='Blues', alpha=0.7, edgecolors='grey', linewidths=0.3
)
plt.colorbar(scatter, ax=axes[1], label='Transport Gap Index')
axes[1].set_xlabel('Office Density Index')
axes[1].set_ylabel('Nighttime Demand Index')
axes[1].set_title('3-Axis Analysis\n(size=priority, color=transport gap)')

# TOP 5 라벨
for _, row in top5.iterrows():
    axes[1].annotate(row['행정동명'],
                     xy=(row['오피스_밀집도_지수'], row['야간수요_지수']),
                     fontsize=7, ha='left',
                     xytext=(4, 4), textcoords='offset points')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '04_priority_map.png', dpi=150, bbox_inches='tight')
plt.close()
print('우선순위 지도 저장 완료 → output/04_priority_map.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


우선순위 지도 저장 완료 → output/04_priority_map.png


In [6]:
# ── 막대 그래프: Top 10 지역 지수 분해 (cell 12에서 더 상세히 다룸)
top10 = df.sort_values('우선순위_지수', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(top10))
w = 0.25

ax.bar(x - w, top10['오피스_밀집도_지수'],  w, label=f'Office Density ({W_OFFICE*100:.0f}%)',  color='#1F4E79')
ax.bar(x,     top10['대중교통공백_지수'],    w, label=f'Transport Gap ({W_TRANSPORT*100:.0f}%)', color='#2E75B6')
ax.bar(x + w, top10['야간수요_지수'],        w, label=f'Night Demand ({W_DEMAND*100:.0f}%)',  color='#9DC3E6')

line = ax.plot(x, top10['우선순위_지수'], 'r-o', markersize=6, label='Priority Index', zorder=5)
ax.set_xticks(x)
ax.set_xticklabels(top10['행정동명'], rotation=30, ha='right', fontsize=9)
ax.set_title('Top 10 Priority Dongs — Index Breakdown', fontsize=14)
ax.set_ylabel('Index (0~1)')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '04_priority_breakdown.png', dpi=150, bbox_inches='tight')
plt.close()
print('Top 10 막대 그래프 저장 → output/04_priority_breakdown.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


Top 10 막대 그래프 저장 → output/04_priority_breakdown.png


## 4. 결과 검증: 잠재 수요 역산

In [7]:
# ── 잠재 수요 역산 (실제 종사자 수 데이터 활용)
# 공식: 오피스 종사자수 × 야근율 × 막차이후_택시의존율 = 잠재 수요(건/일)
OVERWORK_RATE    = 0.25   # 평일 야근 비율 25% (고용부 실태조사)
TAXI_DEPEND_RATE = 0.30   # 막차 이후 택시 의존율 30%

df_valid = df.copy()
df_valid['잠재수요_건일'] = (
    df_valid['오피스_종사자수'].fillna(0) * OVERWORK_RATE * TAXI_DEPEND_RATE
).round(0).astype(int)

GANGNAM_BASELINE = 638   # 강남 시범 운행 기준치
df_valid['강남대비_배율'] = (df_valid['잠재수요_건일'] / GANGNAM_BASELINE).round(2)

print('=== 결과 검증: 잠재 수요 역산 ===')
print(f'  야근율 가정:           {OVERWORK_RATE*100:.0f}%')
print(f'  막차 후 택시 의존율:   {TAXI_DEPEND_RATE*100:.0f}%')
print(f'  강남 시범 기준치:      {GANGNAM_BASELINE}건/일')
print()
top5_verify = df_valid.sort_values('우선순위_지수', ascending=False).head(5)
print(top5_verify[['순위','행정동명','오피스_종사자수','잠재수요_건일','강남대비_배율','우선순위_지수']].to_string(index=False))


=== 결과 검증: 잠재 수요 역산 ===
  야근율 가정:           25%
  막차 후 택시 의존율:   30%
  강남 시범 기준치:      638건/일

 순위 행정동명  오피스_종사자수  잠재수요_건일  강남대비_배율  우선순위_지수
  1  여의동    136466    10235    16.04 0.655221
  2  가산동    106631     7997    12.53 0.569220
  3 역삼1동    105551     7916    12.41 0.565308
  4 논현2동     36372     2728     4.28 0.560421
  5 수유2동       544       41     0.06 0.485006


In [8]:
# ── TOP 10 막대 그래프: 지수 분해
top10 = df_valid.sort_values('우선순위_지수', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 지수 분해
x = np.arange(len(top10))
w = 0.25
axes[0].bar(x - w, top10['오피스_밀집도_지수'],  w, label=f'Office Density ({W_OFFICE*100:.0f}%)',  color='#1F4E79')
axes[0].bar(x,     top10['대중교통공백_지수'],    w, label=f'Transport Gap ({W_TRANSPORT*100:.0f}%)', color='#2E75B6')
axes[0].bar(x + w, top10['야간수요_지수'],        w, label=f'Night Demand ({W_DEMAND*100:.0f}%)',  color='#9DC3E6')

line = axes[0].plot(x, top10['우선순위_지수'], 'r-o', markersize=6, label='Priority Index', zorder=5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(top10['행정동명'], rotation=30, ha='right', fontsize=9)
axes[0].set_title('Top 10 Priority Dongs — Index Breakdown')
axes[0].set_ylabel('Index (0~1)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.05)

# 오른쪽: 잠재 수요
axes[1].barh(top10['행정동명'][::-1], top10['잠재수요_건일'][::-1], color='tomato', alpha=0.8)
axes[1].set_xlabel('Estimated Potential Demand (trips/day)')
axes[1].set_title('Top 10 — Potential Demand Estimate')
axes[1].axvline(x=GANGNAM_BASELINE, color='navy', linestyle='--', label=f'Gangnam baseline ({GANGNAM_BASELINE})')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '04_top10_breakdown.png', dpi=150, bbox_inches='tight')
plt.close()
print('Top 10 분석 차트 저장 완료 → output/04_top10_breakdown.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


Top 10 분석 차트 저장 완료 → output/04_top10_breakdown.png


## 5. 최종 결과 저장

In [9]:
# ── 전체 결과 CSV 저장 (구역유형 컬럼은 분류 셀 실행 후 자동 포함)
df_result = df_valid.drop(columns='geometry').sort_values('순위')
df_result.to_csv(OUTPUT_DIR + 'stage04_priority_result.csv', index=False, encoding='utf-8-sig')
print('최종 결과 저장 완료 → output/stage04_priority_result.csv')

print()
print('=== 분석 완료 요약 ===')
print(f'분석 대상 행정동 수: {len(df_result)}개')
for i in range(min(5, len(df_result))):
    row = df_result.iloc[i]
    print(f'Top {i+1}: {row["행정동명"]} (지수: {row["우선순위_지수"]:.3f}, 잠재수요: {row["잠재수요_건일"]:.0f}건/일)')
print()
print('컬럼:', list(df_result.columns))
print('출처: 통계청 사업체조사(1단계) / 서울시 버스노선·정류소정보(2단계) / KT 심야유동인구(3단계, 빅데이터캠퍼스)')


최종 결과 저장 완료 → output/stage04_priority_result.csv

=== 분석 완료 요약 ===
분석 대상 행정동 수: 426개
Top 1: 여의동 (지수: 0.655, 잠재수요: 10235건/일)
Top 2: 가산동 (지수: 0.569, 잠재수요: 7997건/일)
Top 3: 역삼1동 (지수: 0.565, 잠재수요: 7916건/일)
Top 4: 논현2동 (지수: 0.560, 잠재수요: 2728건/일)
Top 5: 수유2동 (지수: 0.485, 잠재수요: 41건/일)

컬럼: ['행정동코드', '행정동명', '오피스_종사자수', '오피스_밀집도_지수', '대중교통공백_지수', '심야유동인구', '야간수요_지수', '우선순위_지수', '순위', '잠재수요_건일', '강남대비_배율']
출처: 통계청 사업체조사(1단계) / 서울시 버스노선·정류소정보(2단계) / KT 심야유동인구(3단계, 빅데이터캠퍼스)


In [10]:
# ── 구역 유형 분류: 오피스 밀집형 vs 교통공백형
#
# ■ 1유형 (오피스 밀집형): 오피스_밀집도_지수 >= 0.2
#   → 심야 야근 귀가 수요 높음 / 잠재수요 크고 수익성 있음 / 1차 확대 대상
#
# ■ 2유형 (교통공백형): 오피스_밀집도_지수 < 0.1 AND 대중교통공백_지수 >= 0.7
#   → 올빼미버스 미커버 주거지역 / 심야 이동수단 자체가 없음 / 사회적 형평성 관점
#
# ■ 복합형: 0.1 <= 오피스_밀집도_지수 < 0.2 (소규모 업무+교통공백 혼재)

df_valid['구역유형'] = '일반'
df_valid.loc[df_valid['오피스_밀집도_지수'] >= 0.2, '구역유형'] = '오피스 밀집형'
df_valid.loc[
    (df_valid['오피스_밀집도_지수'] < 0.1) &
    (df_valid['대중교통공백_지수'] >= 0.7),
    '구역유형'
] = '교통공백형'
df_valid.loc[
    (df_valid['오피스_밀집도_지수'] >= 0.1) &
    (df_valid['오피스_밀집도_지수'] < 0.2) &
    (df_valid['대중교통공백_지수'] >= 0.7),
    '구역유형'
] = '복합형'

type_counts = df_valid['구역유형'].value_counts()
print('=== 구역 유형 분포 ===')
print(type_counts)
print()

# 유형별 TOP 구역
for typ in ['오피스 밀집형', '교통공백형', '복합형']:
    subset = df_valid[df_valid['구역유형'] == typ].sort_values('우선순위_지수', ascending=False).head(10)
    print(f'\n■ {typ} TOP 10')
    print(subset[['순위','행정동명','오피스_밀집도_지수','대중교통공백_지수','야간수요_지수','우선순위_지수']].to_string(index=False))


=== 구역 유형 분포 ===
구역유형
일반         300
교통공백형      102
오피스 밀집형     23
복합형          1
Name: count, dtype: int64


■ 오피스 밀집형 TOP 10
 순위 행정동명  오피스_밀집도_지수  대중교통공백_지수  야간수요_지수  우선순위_지수
  1  여의동    1.000000   0.743700 0.149755 0.655221
  2  가산동    0.781204   0.690931 0.179907 0.569220
  3 역삼1동    0.773284   0.231402 0.712228 0.565308
  4 논현2동    0.265958   1.000000 0.391119 0.560421
 14 삼성2동    0.235846   0.691353 0.448957 0.459207
 20 구로3동    0.422661   0.418749 0.507660 0.446791
 33 양재1동    0.200645   0.940235 0.075795 0.422047
 53 가양1동    0.345798   0.774687 0.000000 0.392170
 59 잠실6동    0.226775   0.565932 0.345814 0.381192
 94  상암동    0.210876   0.724828 0.065587 0.347172

■ 교통공백형 TOP 10
 순위    행정동명  오피스_밀집도_지수  대중교통공백_지수  야간수요_지수  우선순위_지수
  5    수유2동    0.003212   1.000000 0.446271 0.485006
  6     성현동    0.003175   0.988677 0.455016 0.483653
  7    삼각산동    0.002171   1.000000 0.430229 0.479829
  8     삼전동    0.018561   0.992517 0.402672 0.474679
  9    개봉3동    0.040078   0.935690 0.41504

In [11]:
# ── 유형별 시각화
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.patches import Patch

color_map = {'오피스 밀집형': '#C00000', '교통공백형': '#2E75B6', '복합형': '#7030A0', '일반': '#D9D9D9'}

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# 왼쪽: 유형별 지도
gdf_type = gpd.GeoDataFrame(
    df_valid[['행정동코드','구역유형','우선순위_지수','geometry']].copy(),
    geometry='geometry', crs='EPSG:4326'
)
gdf_type['color'] = gdf_type['구역유형'].map(color_map)
gdf_type.plot(color=gdf_type['color'], ax=axes[0], linewidth=0.3, edgecolor='white')

legend_elements = [
    Patch(facecolor='#C00000', label='Office-Dense (1st priority)'),
    Patch(facecolor='#7030A0', label='Mixed Type'),
    Patch(facecolor='#2E75B6', label='Transport Gap (equity target)'),
    Patch(facecolor='#D9D9D9', label='General'),
]
axes[0].legend(handles=legend_elements, loc='lower left', fontsize=8)

# TOP 5 오피스 밀집형 라벨
top5_office = df_valid[df_valid['구역유형']=='오피스 밀집형'].sort_values('우선순위_지수', ascending=False).head(5)
for _, row in top5_office.iterrows():
    cx, cy = row['geometry'].centroid.x, row['geometry'].centroid.y
    axes[0].annotate(row['행정동명'], xy=(cx,cy), fontsize=7, ha='center',
                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec='none'))

axes[0].set_title('Autonomous Taxi Expansion Zone Classification')
axes[0].set_axis_off()

# 오른쪽: 유형별 TOP 5 막대 비교
types_to_show = ['오피스 밀집형', '교통공백형']
y_positions = []
y_labels = []
colors_bar = []
index_vals = []
y = 0

for typ in types_to_show:
    subset = df_valid[df_valid['구역유형']==typ].sort_values('우선순위_지수', ascending=False).head(5)
    for _, row in subset.iterrows():
        y_labels.append(f"{row['행정동명']} ({typ[:3]})")
        index_vals.append(row['우선순위_지수'])
        colors_bar.append(color_map[typ])
        y += 1

axes[1].barh(range(len(y_labels)), index_vals[::-1], color=colors_bar[::-1], alpha=0.85)
axes[1].set_yticks(range(len(y_labels)))
axes[1].set_yticklabels(y_labels[::-1], fontsize=9)
axes[1].set_xlabel('Priority Index (0~1)')
axes[1].set_title('Top 5 by Type — Office Dense vs Transport Gap')
axes[1].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# 범례
legend2 = [Patch(facecolor='#C00000', label='Office Dense'),
           Patch(facecolor='#2E75B6', label='Transport Gap')]
axes[1].legend(handles=legend2, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + '04_zone_classification.png', dpi=150, bbox_inches='tight')
plt.close()
print('구역 유형 분류 지도 저장 완료 → output/04_zone_classification.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


구역 유형 분류 지도 저장 완료 → output/04_zone_classification.png
